In [1]:
################### Processing csv files to get points values! ########

from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyproj import Proj, transform

In [2]:
def count_non_nan_values(array):
        """Counts non-NaN values in an array while ignoring non-numeric elements."""
        # Filter to keep only numeric data (e.g., ints and floats)
        numeric_values = [x for x in array if isinstance(x, (int, float))] 
        # Convert to NumPy array and count non-NaN values
        return np.sum(~np.isnan(np.array(numeric_values, dtype=float)))
    
def data_prep(path, file, year, month1, month2, cutoff=0, direction=0, cutlon=0, forcevalue=0, skvalue=0, sensor=0):
    ####### filter the data with 015042 index ######
    df = pd.read_csv(path + file)
    df_filtered = df[~df['system:index'].str.contains('_015042_')]
    df_filtered['ID'] = range(0, len(df_filtered))  # Generates IDs starting from 1
    
    ####### get list arranged by dates ######
    unique_dates = sorted(df_filtered['date'].unique())
    unique_dates_series = {}
    unique_dates_indices = {}
    
    for name in unique_dates:
        # Find the indices where 'name' matches the current unique value
        indices = df_filtered.index[df_filtered['date'] == name].tolist()
        # Store the indices in the dictionary with the unique name as the key
        unique_dates_indices[name] = indices
        if sensor == 0:
            dates_data = df_filtered[df_filtered['date'] == name][['lat', 'lon', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
        else:
            dates_data = df_filtered[df_filtered['date'] == name][['lat', 'lon', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
        unique_dates_series[name] = dates_data

    ####### selected the months ######
    keys = list(unique_dates_series.keys())
    #values_2013_04_02 = unique_dates_series[keys[0]]
    
    date_vector = [datetime.strptime(date, '%Y-%m-%d') for date in keys]
    start_date = datetime(year, month1, 1)
    if month2 != 2:
        end_date = datetime(year, month2, 30)
    else:
        end_date = datetime(year, month2, 28)
    # Get indices for dates within the specified range
    indices_in_range = [i for i, date in enumerate(date_vector) if start_date <= date <= end_date]
    
    selected_keys = [keys[i] for i in indices_in_range]
    
    key_with_max_non_nan_arrays_index = max(
        range(len(selected_keys)), 
        key=lambda i: count_non_nan_values(unique_dates_series[selected_keys[i]])
    )
    print("the index with the most # of arrays is:")
    print(key_with_max_non_nan_arrays_index)
    
    if not forcevalue:
        sk = selected_keys[key_with_max_non_nan_arrays_index] ### only select the first element
    else:
        sk = selected_keys[skvalue] 
    print('sk is')
    print(sk)
    #first_selected_key = selected_keys[0] ### only select the first element
    
    # if not isinstance(selected_keys, list):
    #     selected_keys = [selected_keys] 
    ####### get training data and filter out MZ ######
    #training_data = [unique_dates_series[key] for key in sk]
    training_data = unique_dates_series[sk] 
    predictor = []
    target = []
    coord = []
    # Loop through each (1585, 10) array in training_data
    print('training data is:')
    #print(training_data)
    print(training_data.shape)
    #for array in training_data:
        # Get rows where the target is not 'TZ'
    mask = training_data[:, 11] != 'MZ'
    
    predictor.append(training_data[mask, 2:11])  # First 9 columns
    target.append(training_data[mask, 11])       # Target column
    coord.append(training_data[mask, :2])
    
    # Convert to NumPy arrays if desired
    predictor = np.array(predictor)
    target = np.array(target)
    coord = np.array(coord)
    
    X = predictor.reshape(-1, 9)  # Shape: (5 * 1585, 9)
    y = target.reshape(-1)        # Shape: (5 * 1585,)
    coord = coord.reshape(-1, 2)
    ### cleanse the data by removing nan ####
    valid_indices = []

    # Loop through each row and check for NaN in the first column
    for i in range(X.shape[0]):
        if not np.isnan(X[i, 0]):
            valid_indices.append(i)
    
    # Create new X and y with only valid rows
    X = X[valid_indices]
    y = y[valid_indices]
    coord = coord[valid_indices]

    print('the size of dataset is:')
    print(X.shape[0])
    #print(X)

    if cutoff:
        if direction > 0: ### keep the eastern side
            masklon = coord[:, 1] > cutlon
        else:
            masklon = coord[:, 1] <= cutlon
    
        X = X[masklon]
        y = y[masklon]
        coord = coord[masklon]
    
        print('the size of lon filtered dataset is:')
        print(X.shape[0])
        #print(X)
    ### remove duplicated points
    # X = np.array(X, dtype=float)
    # y = np.array(y, dtype='str')
    # coord = np.array(coord, dtype=float)
    # ## duplicated data clean ##
    # _, unique_indices = np.unique(coord, axis=0, return_index=True)

    # # Sort indices to maintain the original order (optional but useful)
    # sorted_indices = np.sort(unique_indices)
    
    # # Step 2: Apply the mask to remove duplicates from miscoord
    # X = X[sorted_indices]
    # y = y[sorted_indices]
    # coord = coord[sorted_indices]
    # print("this includes that much of rows:")
    # print(X.shape[0])
    
    return X, y, coord

In [3]:
def model_train_test(X, y):
    # Initialize the RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    
    # Fit the model to the reshaped data
    model.fit(X, y)
    
    # After training, the model can be used for predictions or evaluation
    y_pred = model.predict(X)
    
    # Calculate accuracy by comparing the predictions with the true labels (y)
    accuracy = model.score(X, y)
    
    print(f"Training and testing using the whole dataset, accuracy is: {accuracy * 100:.2f}%")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    
    # Optionally, evaluate the model on the test set
    score = model.score(X_test, y_test)
    print(f"Testing using 20% of the whole dataset, accuracy is: {score*100:.2f}%")
    
    y_pred = model.predict(X_test)

    print("the number of elements for the prediction:")
    print(y_pred.shape[0])
    cm = confusion_matrix(y_test, y_pred, labels=['VZ', 'TZ', 'WZ'])
    
    # Plot the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VZ', 'TZ', 'WZ'])
    disp.plot(cmap='Blues', values_format='.0f')
    
    return accuracy

In [4]:
def model_across_years(X, y, X_test, y_test, coord_test):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)
    score = model.score(X_test, y_test)
    print(f"Testing across years, accuracy is: {score*100:.2f}%")
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=['VZ', 'TZ', 'WZ'])
    
    # Plot the confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['VZ', 'TZ', 'WZ'])
    disp.plot(cmap='Blues', values_format='.0f')

    incorrect_indices = y_test != y_pred  # Boolean array where predictions are wrong
    misclassified_data = coord_test[incorrect_indices]  # Coordinates of misclassified samples
    misclassified_y_true = y_test[incorrect_indices]  # True values of misclassified samples
    misclassified_y_pred = y_pred[incorrect_indices]  # Predicted values of misclassified samples
    
    # Output misclassified elements
    print("\nMisclassified samples:")
    # for i in range(len(misclassified_data)):
    #     print(f"Coord: {misclassified_data[i]}, Predicted: {misclassified_y_pred[i]}, True: {misclassified_y_true[i]}")
    return misclassified_data, misclassified_y_pred, misclassified_y_true

In [5]:
def export_mispred_to_kml(coord, mispred, output_path):
    # Define colors for each class
    color_mapping = {
        'VZ': 'ff00ff00',  # Green in KML color format (aabbggrr)
        'WZ': 'ff0000ff',  # Red
        'TZ': 'ff00ffff'   # Yellow
    }
    
    # Create a KML object
    kml = simplekml.Kml()
    
    # Iterate through the coordinates and mispred values
    for (lat, lon), label in zip(coord, mispred):
        # Add a point to the KML
        pnt = kml.newpoint(coords=[(lon, lat)])
        pnt.style.iconstyle.color = color_mapping[label]
        pnt.style.iconstyle.scale = 1.0  # Icon size
        pnt.name = label  # Label as the name of the point
    
    # Save the KML file
    kml.save(output_path)
    print(f"KML file saved at: {output_path}")

In [ ]:
X_2009, y_2009, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2009/', 'class_2009.csv', 2009, 12, 12, 0, 0, 0, 0, 0, 1) ## mixing 2022 Mar and 2023 Feb 

In [7]:
#X_1985, y_1985, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/1985/', 'class_1985_shadow.csv', 1986, 1, 1, 0, 0, 0, 0, 0, 1) ## mixing 2022 Mar and 2023 Feb
X_1986, y_1986, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/1985/', 'class_1985_shadow.csv', 1986, 2, 2, 0, 0, 0, 0, 0, 1) ## mixing 2022 Mar and 2023 Feb 

/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_66705/1074313299.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['ID'] = range(0, len(df_filtered))  # Generates IDs starting from 1


the index with the most # of arrays is:
0
sk is
1986-02-22
training data is:
(1579, 12)
the size of dataset is:
1054


In [ ]:
#X_2022, y_2022, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2023/', 'class_2023_train.csv', 2022, 3, 3, 0, 0, 0, 0, 0) ## mixing 2022 Mar and 2023 Feb 
#X_2023, y_2023, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2023/', 'class_2023_train.csv', 2023, 2, 2, 0, 0, 0, 1, 1) ## mixing 2022 Mar and 2023 Feb 

In [ ]:
X_2022, y_2022, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2023/', 'class_2023_train.csv', 2022, 3, 3, 1, -1, -80.528389,0) ## mixing 2022 Mar and 2023 Feb 


In [ ]:
X_2023, y_2023, coord = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2023/', 'class_2023_train.csv', 2023, 2, 2, 1, 1, -80.528389,1,1) ## mixing 2022 Mar and 2023 Feb 

In [ ]:
X_2013, y_2013, coord_2013 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2013/', '2013_pts.csv', 2013, 3, 4, 0, 0, 0, 0)

In [ ]:
X_2018, y_2018, coord_2018 = data_prep('/Users/bz294/Documents/WhiteZone/Points/KML/2018/', 'class2018_pts.csv', 2017, 12, 12, 0, 0, 0, 0)

In [ ]:
X_combined = np.vstack((X_2022, X_2023))
y_combined = np.vstack((y_2022.reshape(-1, 1), y_2023.reshape(-1, 1)))
accuracy = model_train_test(X_combined, y_combined)

In [ ]:
accuracy = model_train_test(X_2009, y_2009)

In [ ]:
accuracy = model_train_test(X_1985, y_1985)

In [ ]:
accuracy = model_train_test(X_2018, y_2018)

In [ ]:
miscoord, mispred, mistrue = model_across_years(X_combined, y_combined, X_2013, y_2013, coord)


In [ ]:
miscoord, mispred, mistrue = model_across_years(X, y, X_2018, y_2018, coord_2018)

In [ ]:
##### export to kml 

export_mispred_to_kml(miscoord, mispred, "/Users/bz294/Documents/WhiteZone/Points/KML/2018/class2013_pred_new_1.kml")

In [ ]:
export_mispred_to_kml(miscoord, mistrue, "/Users/bz294/Documents/WhiteZone/Points/KML/2018/class2013_true_new_1.kml")

In [9]:
################ Making predictions for the region #################
feature = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/1985/feat_19860222.npy')
coord = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/1985/coord_19860222.npy')

#feature_2022 = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/2023/feat_all_2022.npy')
#coord_2022 = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/2023/coord_all_2022.npy')

#feature_2023 = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/2023/feat_all_2023.npy')
#coord_2023 = np.load('/Users/bz294/Documents/WhiteZone/Points/Regional/2023/coord_all_2023.npy')

In [10]:
#### predictions ###
model = RandomForestClassifier(n_estimators=100, random_state=42)
    
# Fit the model to the reshaped data
model.fit(X_1986, y_1986)

# After training, the model can be used for predictions or evaluation
y_region = model.predict(feature)

In [11]:
geometry = [Point(xy) for xy in coord]

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame({'y_region': y_region}, geometry=geometry)

# Set the CRS to UTM 17N (EPSG:32617)
gdf.set_crs('EPSG:32617', inplace=True)

# Save to a shapefile
output_shapefile = 'output_points_1986.shp'  # Define your output shapefile path
gdf.to_file(output_shapefile)

print(f'Shapefile saved to: {output_shapefile}')

Shapefile saved to: output_points_1986.shp


In [ ]:
color_map = {
    'WZ': 'grey',
    'VZ': 'red',
    'TZ': 'green'
}

# Map the colors to the y_region column
gdf['color'] = gdf['y_region'].map(color_map)

# Plot with different colors for each class
fig, ax = plt.subplots(figsize=(10, 10))
for region, group in gdf.groupby('y_region'):
    group.plot(ax=ax, color=group['color'], marker='o', markersize=5, label=region)

# Add legend and title
ax.legend(title='Region Classes')
ax.set_title('Visualization of Shapefile by y_region Classes', fontsize=16)
plt.show()

In [ ]:
#### predictions ###
model = RandomForestClassifier(n_estimators=100, random_state=42)
    
# Fit the model to the reshaped data
model.fit(X_2022, y_2022)

# After training, the model can be used for predictions or evaluation
y_region_2022 = model.predict(feature_2022)

model = RandomForestClassifier(n_estimators=100, random_state=42)
    
# Fit the model to the reshaped data
model.fit(X_2023, y_2023)
y_region_2023 = model.predict(feature_2023)

# model.fit(X_2023, y_2023)

# y_2023 = model.predict(feature_2023)

In [ ]:
##### Make point shapefile #######
threshold = 547396.0698541238 ## the cutting line

geometry_2022 = [Point(xy) for xy in coord_2022]
filtered_geo_2022 = [point for point in geometry_2022 if point.x <= threshold]
filtered_y_2022 = [y for point, y in zip(geometry_2022, y_region_2022) if point.x <= threshold]

# Print the number of filtered y values
print(f"Number of filtered_y_2022: {len(filtered_y_2022)}")

geometry_2023 = [Point(xy) for xy in coord_2023]
filtered_geo_2023 = [point for point in geometry_2023 if point.x > threshold]
filtered_y_2023 = [y for point, y in zip(geometry_2023, y_region_2023) if point.x > threshold]

# Print the number of filtered y values
print(f"Number of filtered_y_2023: {len(filtered_y_2023)}")

geometry = filtered_geo_2022 + filtered_geo_2023
y_region = filtered_y_2022 + filtered_y_2023

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame({'y_region': y_region}, geometry=geometry)

# Set the CRS to UTM 17N (EPSG:32617)
gdf.set_crs('EPSG:32617', inplace=True)

# Save to a shapefile
output_shapefile = 'output_points_2023_final.shp'  # Define your output shapefile path
gdf.to_file(output_shapefile)

print(f'Shapefile saved to: {output_shapefile}')

In [ ]:
from pyproj import Proj, transform
utm_proj = Proj(proj='utm', zone=17, ellps='WGS84', south=False)

# Example longitude and latitude (you need both for the conversion)
longitude = -80.528389  # Replace with your longitude
latitude = 25.5    # Replace with your latitude

# Convert to UTM coordinates
x, y = utm_proj(longitude, latitude)

print(f"Longitude {longitude}, Latitude {latitude} converts to UTM x: {x}, y: {y}")


In [ ]:
####### for 2022/2023 ##################


geometry = [Point(xy) for xy in coord_2022]

# Create a GeoDataFrame
gdf = gpd.GeoDataFrame({'y_region': y_region}, geometry=geometry)

# Set the CRS to UTM 17N (EPSG:32617)
gdf.set_crs('EPSG:32617', inplace=True)

# Save to a shapefile
output_shapefile = 'output_points.shp'  # Define your output shapefile path
gdf.to_file(output_shapefile)

print(f'Shapefile saved to: {output_shapefile}')

In [ ]:
############# Testing using 2013 data ################

start_date = datetime(2013, 2, 1)
end_date = datetime(2013, 4, 30)

# Get indices for dates within the specified range
indices_in_range = [i for i, date in enumerate(date_vector) if start_date <= date <= end_date]

selected_keys = [keys[i] for i in indices_in_range]

# Step 3: Retrieve data from unique_dates_series using the selected keys
testing_data = [unique_dates_series[key] for key in selected_keys]

predictor_test = []
target_test = []
# Loop through each (1585, 10) array in training_data
for array in testing_data:

    mask = array[:, 11] != 'TZ'
    # Slice the first 9 columns and append to first_9_columns list
    predictor_test.append(array[mask, 2:11])  # First 9 columns
    target_test.append(array[mask, 11])       # Target column

# Convert to NumPy arrays if desired
predictor_test = np.array(predictor_test)
target_test = np.array(target_test)

In [ ]:
selected_keys

In [ ]:
############ make predictions ########

X = predictor_test.reshape(-1, 9)  # Shape: (5 * 1585, 9)
y = target_test.reshape(-1)        # Shape: (5 * 1585,)

y_pred_test = model.predict(X)

# Calculate accuracy by comparing the predictions with the true labels (y)
accuracy = model.score(X, y)

print(f"Training accuracy: {accuracy * 100:.2f}%")

In [ ]:
predictor_test.shape

In [ ]:
target_test.shape

In [ ]:
len(testing_data) * testing_data[0].shape[0]

In [ ]:
###### know the indices of the wrong predictions ####

y = y.reshape(target_test.shape[0], predictor_test.shape[1])
y_pred_test = y_pred_test.reshape(target_test.shape[0], predictor_test.shape[1])

# Find where predictions don't match the true labels
incorrect_mask = y_pred_test != y

# Get the indices of incorrect predictions for each sample
incorrect_indices = np.argwhere(incorrect_mask) ## these are the indices from the subset without TZ, not all!!

# Calculate accuracy
accuracy = model.score(X, y.reshape(-1))
print(f"Training accuracy: {accuracy * 100:.2f}%")

# Print incorrect indices
print("Indices of incorrect predictions (in 5 x 1371 format):")
print(incorrect_indices)

In [ ]:
###### get lat and lon for "incorrect" predictions #######
lat = []
lon = []
select_scene = 0
first = training_data[0]
mask = first[:, 11] != 'TZ'
filtered_first = first[mask]

# Apply mask to select only rows without 'TZ' in the target column
lon.append(filtered_first[:, 1])       # First 9 columns
lat.append(filtered_first[:, 0])       # Target column

lat = np.array(lat).flatten()
lon = np.array(lon).flatten()
filtered_indices = incorrect_indices[incorrect_indices[:, 0] == select_scene, 1] ## select the third one, which is March, 2013 according to GE. 
filtered_indices = np.array(filtered_indices)
#lat[filtered_indices]
#lon[filtered_indices]
#target_test[0, filtered_indices] -- ground truth
#y_pred_test[0, filtered_indices] -- predicted 

In [ ]:
incorrect_indices

In [ ]:
y_pred_test[0, :]

In [ ]:
######### generating kml ######################

kml = simplekml.Kml()

# Define colors for each 'Description' value
colors = {
    'VZ': 'ff00ff00',  # Green in KML color format (aabbggrr)
    'WZ': 'ff0000ff',  # Red
}

values = y_pred_test[0, filtered_indices] 
for lattemp, lontemp, val in zip(lat[filtered_indices], lon[filtered_indices], y_pred_test[select_scene, filtered_indices]): ## pay attention to change the index here to 2. 
    color = colors.get(val, 'ff000000')  # Default to black if value not in colors
    pnt = kml.newpoint(coords=[(lontemp, lattemp)])  # KML uses (longitude, latitude)
    pnt.style.iconstyle.color = color  # Set the color based on the value

# Save the KML file
kml.save("/Users/bz294/Documents/WhiteZone/Points/KML/model_check2_2013.kml")

In [ ]:
values

In [ ]:
filtered_indices

In [ ]:
len(lat[0])

In [ ]:
name_1_data = unique_name_series.get(1)

# Display the data
type(name_1_data)
name_1_indices = unique_name_indices.get(1)
len(name_1_indices)

In [ ]:
##### Get time stamps #######
date_values = df_filtered.loc[name_1_indices, 'date'].values

np.unique(date_values).shape
date_values
date_times = np.array([datetime.strptime(date, '%m/%d/%y') for date in date_values])

days_ind = (date_times - date_times[0]).astype('timedelta64[D]').astype(int)

# Convert the first element to 0 and keep the rest
days_ind[0] = 0

# Display the result
print(days_ind)

In [ ]:
date_times

In [ ]:
date_strings

In [ ]:
########### Plot time series of a random pixel ####################
%matplotlib inline
b2 = name_1_data[:, 0] 
ndvi = name_1_data[:, 7] 

plt.figure(figsize=(10, 7))
plt.plot(days_ind, ndvi, marker='o', linestyle='-')
plt.title('Time Series Plot')
plt.xlabel('Date')
plt.ylabel('Value')
xticks = range(0, max(days_ind) + 1, 500)

date_strings = [date_times[0] + timedelta(days=500 * i) for i in range(9)]
date_strings = [date.strftime('%Y-%m-%d') for date in dates]

plt.xticks(ticks=xticks, labels=date_strings, rotation=45)

plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.2)
plt.show()

In [ ]:
############# model building ###############




In [ ]:
xticks_indices

In [ ]:
len(df)

In [ ]:
unique_dates

In [ ]:
50100 / 981

In [ ]:
################ Plot time series for selected pixels ####################

name_1_data = unique_name_series.get(1)

In [ ]:
unique_dates

In [ ]:
#path = '/Users/bz294/Documents/WhiteZone/Points/'
#file1 = 'RevisedTimeSeries2__.csv'


#df_filtered = df

#coord = pd.read_csv('/Users/bz294/Documents/WhiteZone/Points/points_coord2.csv')

# Display the first few rows of the DataFrame
#print(coord.head())

#df_filtered = pd.merge(df_filtered, coord[['ID', 'lat', 'lon']], on='ID', how='left') ## no need to merge

# unique_names = sorted(df_filtered['ID'].unique())
# unique_name_series = {}
# unique_name_indices = {}

# for name in unique_names:
#     # Find the indices where 'name' matches the current unique value
#     indices = df_filtered.index[df_filtered['ID'] == name].tolist()
#     # Store the indices in the dictionary with the unique name as the key
#     unique_name_indices[name] = indices
#     name_data = df_filtered[df_filtered['ID'] == name][['lat', 'lon', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'ndmi', 'ndvi', 'savi', 'descriptio']].to_numpy()
#     unique_name_series[name] = name_data